# M14, 60 km/h ramp: direct SUMO-PPO trained longer (1000 → 1200 episodes)

Continues the three finished direct SUMO-PPO runs (policy seeds 0, 1, 2) in the same Drive folder (`m14_ramp60`)
from 1000 to 1200 training episodes, so direct PPO gets at least as much compute as Surrogate-PPO:

| | direct PPO, 1000 episodes | direct PPO, 1200 episodes | Surrogate-PPO (fast code) |
|---|---|---|---|
| SUMO episodes (training + validation) | ≈ 1215 | ≈ 1470 | 908-962 |
| summed compute | ≈ 5.4 h | ≈ 6.6 h | 5.9-6.3 h |

Each run **continues** from its final model (policy, value network, optimiser state and step count; constant
learning rate), so it is the same run trained longer, not a new run. Its validation history and SUMO ledger come
along: the 1200-episode run is charged for all its episodes, and its checkpoint is chosen on validation over the
whole run (steps 0-144k). The 1000-episode results stay as they are.

Then the new policies are evaluated on the test and OOD profiles (everything else is reused) and the tables are
rebuilt: the **SUMO-PPO** row becomes the 1200-episode run, a new **SUMO-PPO (1000 ep.)** row keeps the old one,
and the headline reductions are given against both. The current tables are kept in `tables_direct1000/`.

**Expected time** (12-CPU L4 runtime): about 1 h for the three runs at once (≈ 254 more SUMO episodes each, one at
a time), then 15-20 min of evaluation. Plan for 1.5 h.

**The code is pinned** to commit `@COMMIT@` (`run.py extend-direct`).

| cell | what | when |
|---|---|---|
| 1 | parameters | every session |
| 2 | Drive + code (pinned commit), keeps a copy of the current tables | every session |
| 3 | install SUMO + check | every session |
| 4 | launch / resume in the background | first session, and after a lost session |
| 5 | watchdog: keeps the session busy, relaunches a stopped driver | right after cell 4; leave it running |
| 6 | status | any time (stop cell 5 first, then restart it) |
| 7 | tables | at the end |
| 8 | archive of the results on Drive | at the end |

**After a lost session:** rerun cells 1-3, then cell 4 and cell 5. The runs continue from their latest checkpoint;
an interrupted evaluation is moved aside and redone.

In [ ]:
# 1. Parameters
REPO_URL = "https://github.com/LejunZhou/traffic-surrogate-rl.git"
COMMIT = "39bf227582725412dd75f8d54bfb7fcf7a08da6e"                              # pinned code version
WORK = "/content/drive/MyDrive/m14_ramp60"       # the finished three-seed study
SEEDS = [0, 1, 2]
FROM_BUDGET = 1000                                # finished direct PPO runs, episodes
TO_BUDGET = 1200                                  # continue them to this many training episodes
TORCH_THREADS = 4                                 # CPU threads per PPO process (3 processes share the machine)

In [ ]:
# 2. Drive + code at the pinned commit
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, subprocess, datetime, json

assert os.path.isdir(WORK), f"{WORK} not found"
for s in SEEDS:
    need = f"runs/study/m14/direct_ppo_{FROM_BUDGET}ee_s{s}/final_model.zip"
    assert os.path.exists(f"{WORK}/{need}"), f"{need} missing: the {FROM_BUDGET}-episode run of seed {s} is not finished"
for need in ("runs/study/m14/alinea_tuning.json", "runs/study/m14/arms.json", "runs/study/m14/tables/tables.md"):
    assert os.path.exists(f"{WORK}/{need}"), f"{need} missing"

CLONE = "/content/traffic-surrogate-rl"
if not os.path.isdir(CLONE):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE], check=True)
subprocess.run(["git", "-C", CLONE, "fetch", "--depth", "1", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", CLONE, "checkout", "-q", COMMIT], check=True)

# the current tables and manifest (direct PPO at 1000 episodes), kept once before the rebuild overwrites them
if not os.path.exists(f"{WORK}/runs/study/m14/tables_direct{FROM_BUDGET}"):
    shutil.copytree(f"{WORK}/runs/study/m14/tables", f"{WORK}/runs/study/m14/tables_direct{FROM_BUDGET}")
    shutil.copy2(f"{WORK}/runs/study/m14/arms.json", f"{WORK}/runs/study/m14/arms_direct{FROM_BUDGET}.json")

version_file = f"{WORK}/CODE_VERSION.txt"
if COMMIT[:7] not in open(version_file).read().splitlines()[-1]:
    for item in ["src", "scripts", "configs", "tests", "docs", "run.py", "pyproject.toml", "README.md"]:
        src, dst = f"{CLONE}/m14/{item}", f"{WORK}/{item}"
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
        else:
            shutil.copy2(src, dst)
    with open(version_file, "a") as f:
        f.write(f"{datetime.datetime.now().isoformat(timespec='seconds')}  {COMMIT[:7]}  update (direct PPO {FROM_BUDGET} -> {TO_BUDGET})\n")
print(open(version_file).read())
os.chdir(WORK)
!python run.py extend-direct --help | head -3

In [ ]:
# 3. Install SUMO and dependencies, check the runtime
!pip install -q "eclipse-sumo==1.27.1" "traci==1.27.1" "sumolib==1.27.1" "stable-baselines3>=2.0" "gymnasium>=0.29" pyyaml
import os, sys, sumo, torch
os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
os.environ["PATH"] = os.path.join(os.environ["SUMO_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["MPLBACKEND"] = "Agg"
os.chdir(WORK)
!python run.py check
CORES = os.cpu_count()
WORKERS = max(2, CORES)           # SUMO workers for the final evaluation (each training run simulates one episode at a time)
print(f"\nCPU cores: {CORES} -> evaluation workers {WORKERS}, torch threads per PPO process {TORCH_THREADS}")

## The continuation

Cell 4 starts `run.py extend-direct` as a background process. Logs: `runs/logs/extend_driver.log` (driver) and
`runs/logs/pipeline_direct_s{0,1,2}.log` (the three training runs; earlier sessions' lines are above the new ones).
Cell 4 refuses to start a second driver while one is running.

**Then start cell 5 and leave it running.** Colab disconnects a notebook with no running cell after a while, even if
background processes are busy. Cell 5 keeps a cell running, prints a status line every 5 minutes, and relaunches the
driver (at most 3 times) if it stops before the new tables exist. To run cell 6, stop cell 5 with the stop button
(the runs keep going), then start cell 5 again. Keep the browser tab open and the computer awake.

In [ ]:
# 4. Launch or resume in the background
import os, sys, subprocess
os.makedirs("runs/logs", exist_ok=True)
PID_FILE = "runs/logs/extend_driver.pid"

def driver_alive():
    try:
        pid = int(open(PID_FILE).read().strip())
        os.kill(pid, 0)
        return "Z" not in [l for l in open(f"/proc/{pid}/status") if l.startswith("State:")][0]
    except (OSError, ValueError, IndexError):
        return False              # no pid file, a pid from a previous runtime, or a finished driver

def launch(tag="launch"):
    cmd = [sys.executable, "run.py", "extend-direct", "--seeds", *map(str, SEEDS), "--from-budget", str(FROM_BUDGET),
           "--to-budget", str(TO_BUDGET), "--recover-interrupted", "--workers", str(WORKERS),
           "--torch-threads", str(TORCH_THREADS)]
    log = open("runs/logs/extend_driver.log", "a")
    log.write(f"\n\n===== {tag} " + " ".join(cmd) + "\n"); log.flush()
    proc = subprocess.Popen(cmd, cwd=WORK, stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=dict(os.environ))
    open(PID_FILE, "w").write(str(proc.pid))
    return proc.pid

if driver_alive():
    print("A driver is already running in this runtime; see cell 6.")
else:
    print("driver started, pid", launch())

In [ ]:
# 5. Watchdog: keeps this notebook busy and relaunches the driver if it stops before the tables exist.
#    Leave it running. The stop button ends only this loop; the background runs keep going.
import os, glob, json, time, datetime
os.chdir(WORK)
MAX_RESTARTS, EVERY_S = 3, 300
TABLES = "runs/study/m14/tables/tables.json"

def extension_done():
    try:
        return f"SUMO-PPO ({FROM_BUDGET} ep.)" in json.load(open(TABLES))["table2"]
    except (OSError, ValueError, KeyError):
        return False

def direct_progress(s):
    run = f"runs/study/m14/direct_ppo_{TO_BUDGET}ee_s{s}"
    if os.path.exists(f"{run}/final_model.zip"):
        return "done"
    steps = max((int(p.split("_")[-2]) for p in glob.glob(f"{run}/checkpoints/*_steps.zip")), default=0)
    return f"{steps // 1000}k/{TO_BUDGET * 120 // 1000}k"

restarts = 0
while not extension_done():
    now = datetime.datetime.now(datetime.timezone.utc).strftime("%H:%M UTC")
    if not driver_alive():
        if restarts >= MAX_RESTARTS:
            print(f"{now}  driver stopped {restarts} times; not restarting again. Check the logs (cell 6).")
            break
        restarts += 1
        print(f"{now}  driver not running -> relaunch {restarts}/{MAX_RESTARTS}, pid {launch('launch (watchdog)')}")
    else:
        last = open("runs/logs/extend_driver.log", errors="replace").read().splitlines()[-1:] or [""]
        print(f"{now}  " + ", ".join(f"s{s} {direct_progress(s)}" for s in SEEDS) + f" | {last[0][:70]}")
    time.sleep(EVERY_S)
else:
    print("new tables written - run cells 7 and 8")

In [ ]:
# 6. Status (stop cell 5 first; restart it afterwards)
import os, glob, json
import numpy as np
def tail(path, n=4):
    if os.path.exists(path):
        lines = open(path, errors="replace").read().splitlines()
        print(f"--- {path}"); print("\n".join(lines[-n:]))

print("driver running:", driver_alive())
tail("runs/logs/extend_driver.log", 6)
for s in SEEDS:
    tail(f"runs/logs/pipeline_direct_s{s}.log", 3)
for s in SEEDS:
    run = f"runs/study/m14/direct_ppo_{TO_BUDGET}ee_s{s}"
    print(f"  seed {s}: {direct_progress(s)}")
    npz = f"{run}/eval/evaluations.npz"
    if os.path.exists(npz):
        d = np.load(npz)
        print("    validation mean return by step:", ", ".join(f"{t // 1000}k {r.mean():.1f}" for t, r in zip(d["timesteps"], d["results"])))
    if os.path.exists(f"{run}/eval/selection.json"):
        c = json.load(open(f"{run}/eval/selection.json"))["chosen"]
        print(f"    selected step {c['step']} (validation mean {c['mean']:.1f})")
print("new tables:", extension_done())
errors = [p for p in glob.glob("runs/logs/*.log") if "Traceback" in open(p, errors="replace").read()]
print("logs with a Traceback (older logs may show an old, recovered one):", errors or "none")

In [ ]:
# 7. Tables over seeds 0-2 with both direct PPO budgets (per-seed tables: runs/study/m14/tables/seed_<s>/tables.md)
from IPython.display import Markdown, display
path = "runs/study/m14/tables/tables.md"
display(Markdown(open(path).read()) if extension_done() else Markdown("not built yet (cell 6 shows progress)"))

In [ ]:
# 8. Archive of the results (small files only) next to WORK on Drive
import shutil, datetime, tempfile
keep = ["runs/study/m14/tables", f"runs/study/m14/tables_direct{FROM_BUDGET}", "runs/study/m14/arms.json",
        "runs/ledger", "runs/logs", "runs/commands.jsonl", "reports", "CODE_VERSION.txt"]
for s in SEEDS:
    run = f"runs/study/m14/direct_ppo_{TO_BUDGET}ee_s{s}"
    keep += [f"{run}/eval", f"{run}/continued_from.json", f"{run}/resume_log.jsonl"]
keep += glob.glob("runs/study/m14/eval/*.jsonl") + glob.glob("runs/study/m14/eval/*.summary.json")
stage = tempfile.mkdtemp()
for item in keep:
    if os.path.isdir(item):
        shutil.copytree(item, os.path.join(stage, item), dirs_exist_ok=True)
    elif os.path.exists(item):
        os.makedirs(os.path.dirname(os.path.join(stage, item)) or stage, exist_ok=True)
        shutil.copy2(item, os.path.join(stage, item))
name = f"{WORK}_direct_extend_results_{datetime.datetime.now():%Y%m%d_%H%M}"
print(shutil.make_archive(name, "zip", stage))